In [ ]:
DATASET      = "mosaiks.forest"   
DATASET_ROOT     = "/data/locbench"      
LOC_ENCODER = "satclip"             
SPARSE_MODEL = "splice"             
CONCEPTS_PT  = "../splice_results/geoclip/concept_dictionary_n500000/concepts.pt"  
K            = 10

from pathlib import Path
DATASET_PATH = Path(DATASET_ROOT) / DATASET.replace(".", "/")

In [2]:
import torch, numpy as np
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

def load(path):
    d = torch.load(path, weights_only=False)
    X = (d.get("codes") if "codes" in d else d["embeddings"]).numpy()
    y = (d.get("vals") if "vals" in d else d["values"]).numpy()
    m = ~np.isnan(y); return X[m], y[m]

def run_ridge(prefix):
    X_tr, y_tr = load(DATASET_PATH / f"{prefix}_train.pt")
    X_te, y_te = load(DATASET_PATH / f"{prefix}_test.pt")
    m = RidgeCV().fit(X_tr, y_tr)
    return m, r2_score(y_te, m.predict(X_te))

In [3]:
earth_model, earth_r2 = run_ridge(LOC_ENCODER)
print(f"Earth embeddings ({LOC_ENCODER})  R² = {earth_r2:.4f}")

Earth embeddings (satclip)  R² = 0.6065


In [ ]:
sparse_model, sparse_r2 = run_ridge(SPARSE_MODEL)
print(f"Sparse codes ({SPARSE_MODEL})  R² = {sparse_r2:.4f}")

Sparse codes (splice)  R² = 0.6755


FileNotFoundError: [Errno 2] No such file or directory: 'data/git-10m/concept_dictionaries/concept_dictionary_n500000.json'

In [8]:
concepts = torch.load(CONCEPTS_PT, weights_only=False)
top_idxs = np.argsort(np.abs(sparse_model.coef_))[::-1][:K]
print(f"\nTop {K} concepts:")
for rank, i in enumerate(top_idxs, 1):
    print(f"  {rank:2d}. [{i:4d}] {concepts[i]!r:30s}  w={sparse_model.coef_[i]:+.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'splice_results/geoclip/concept_dictionary_n500000/concepts.pt'